# Monte Carlo Tree Search (MCTS)

Monte Carlo Tree Search (MCTS) is a heuristic search algorithm used for decision-making
in game trees and other sequential decision problems.

It is especially useful when:
- the search space is very large,
- it is impossible to explore the full tree,
- and we can simulate outcomes randomly.

### Main Idea
Instead of exploring the whole tree like MiniMax, MCTS builds the tree gradually
by repeating four steps:

1. **Selection**  
   Start from the root and repeatedly choose the best child until reaching a node
   that is not fully expanded.

2. **Expansion**  
   Add a new child node to the tree.

3. **Simulation**  
   Run a random rollout from the new node until reaching a terminal state.

4. **Backpropagation**  
   Propagate the simulation result back up the tree.

### UCB1 Formula
MCTS often uses the UCB1 rule during selection:

$$
UCB1 = \frac{w_i}{n_i} + c \sqrt{\frac{\ln N}{n_i}}
$$

Where:
- $w_i$: number of wins for child $i$
- $n_i$: number of visits for child $i$
- $N$: number of visits for the parent
- $c$: exploration constant

This balances:
- **exploitation**: choosing moves that already look good
- **exploration**: trying less-visited moves


### Step 1: Define a Simple Game State

To keep the example simple, we use a small number game:

- The game starts at `0`
- Players can add either `1` or `2`
- The player who reaches exactly `5` wins
- If a move goes beyond `5`, it is not allowed

This gives us a small turn-based game suitable for MCTS.


In [1]:
class NumberGameState:
    def __init__(self, total=0, player=1, target=5):
        self.total = total
        self.player = player
        self.target = target

    def get_legal_moves(self):
        moves = []
        for move in [1, 2]:
            if self.total + move <= self.target:
                moves.append(move)
        return moves

    def make_move(self, move):
        return NumberGameState(
            total=self.total + move,
            player=-self.player,
            target=self.target
        )

    def is_terminal(self):
        return self.total == self.target

    def get_winner(self):
        if not self.is_terminal():
            return None
        # The previous player made the winning move
        return -self.player

    def __repr__(self):
        return f"State(total={self.total}, player={self.player})"


### Step 2: Define the MCTS Node

Each tree node stores:
- the current game state
- the parent node
- the move that led to this node
- child nodes
- visit count
- win score
- untried moves

This information is enough to perform the four MCTS phases.


In [2]:
import math
import random


class MCTSNode:
    def __init__(self, state, parent=None, move=None):
        self.state = state
        self.parent = parent
        self.move = move
        self.children = []
        self.visits = 0
        self.wins = 0.0
        self.untried_moves = state.get_legal_moves()

    def is_fully_expanded(self):
        return len(self.untried_moves) == 0

    def best_child(self, exploration_weight=1.41):
        best_score = float("-inf")
        best_node = None

        for child in self.children:
            exploitation = child.wins / child.visits
            exploration = exploration_weight * math.sqrt(
                math.log(self.visits) / child.visits
            )
            score = exploitation + exploration

            if score > best_score:
                best_score = score
                best_node = child

        return best_node

    def expand(self):
        move = self.untried_moves.pop()
        next_state = self.state.make_move(move)
        child_node = MCTSNode(next_state, parent=self, move=move)
        self.children.append(child_node)
        return child_node


### Step 3: Simulation and Backpropagation

Now we implement:
- **rollout**: play random moves until the game ends
- **backpropagate**: update visits and wins from the simulated result

For scoring:
- if the node's player eventually wins, we count it accordingly
- since players alternate, we evaluate the result relative to the root player later


In [3]:
def rollout(state):
    current_state = state

    while not current_state.is_terminal():
        move = random.choice(current_state.get_legal_moves())
        current_state = current_state.make_move(move)

    return current_state.get_winner()


def backpropagate(node, winner, root_player):
    current = node

    while current is not None:
        current.visits += 1

        if winner == root_player:
            current.wins += 1
        elif winner is None:
            current.wins += 0.5

        current = current.parent


### Step 4: MCTS Main Algorithm

This function performs the four MCTS stages repeatedly:

1. **Selection**: move down using UCB1
2. **Expansion**: add one new child
3. **Simulation**: run a random rollout
4. **Backpropagation**: update the path

At the end, we choose the child with the highest number of visits.


In [4]:
def monte_carlo_tree_search(root_state, iterations=1000):
    root_node = MCTSNode(root_state)
    root_player = root_state.player

    for _ in range(iterations):
        node = root_node

        # 1. Selection
        while node.state.is_terminal() is False and node.is_fully_expanded():
            node = node.best_child()

        # 2. Expansion
        if node.state.is_terminal() is False and node.untried_moves:
            node = node.expand()

        # 3. Simulation
        winner = rollout(node.state)

        # 4. Backpropagation
        backpropagate(node, winner, root_player)

    best_move = None
    best_visits = -1

    for child in root_node.children:
        if child.visits > best_visits:
            best_visits = child.visits
            best_move = child.move

    return best_move, root_node


### Step 5: Run MCTS

We start from total = 0.
The algorithm will simulate many random games and estimate which first move is better.

Possible first moves:
- add `1`
- add `2`

The output includes:
- the best move found
- visit count and win score for each child of the root


In [6]:
initial_state = NumberGameState(total=0, player=1, target=5)

best_move, root = monte_carlo_tree_search(initial_state, iterations=1000)

print("Best Move from Root:", best_move)
print("-" * 30)

for child in root.children:
    win_rate = child.wins / child.visits if child.visits > 0 else 0
    print(
        f"Move {child.move} -> Visits: {child.visits}, "
        f"Wins: {child.wins:.1f}, Win Rate: {win_rate:.3f}"
    )


Best Move from Root: 2
------------------------------
Move 2 -> Visits: 631, Wins: 622.0, Win Rate: 0.986
Move 1 -> Visits: 369, Wins: 347.0, Win Rate: 0.940


### Step 6: Interpretation

MCTS does not guarantee the exact optimal move in small numbers of iterations,
but with more simulations it usually becomes more reliable.

In this game:
- some opening moves lead to stronger positions
- MCTS estimates that by repeated random play

The move with the highest visit count is usually selected as the final decision.
